# Try the Jev-class models on a Colab GPU
One model at a time on the GPU. Load it, call it with text, an image and a video, see the response times, switch to another.
A control panel shows every model, what fits this GPU, and Load / Unload / Enable / Disable buttons.

**Two modes.** *Interactive* (default) runs and is called from this notebook. *Serve* (opt-in, step 5) also opens a public
Cloudflare URL and plays an audio loop to keep the session up. Read the note in step 5 before turning it on.

## 1. Get the code

In [1]:
# clones just this folder from GitHub (a sparse clone: the repo holds more than the notebook needs)
REPO = "https://github.com/mvp-scale/workspace"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
SUBDIR = "colab"  #@param {type:"string"}  # leave empty if the repo root is this folder
import os, subprocess

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

sh("rm -rf /content/src")
if SUBDIR:
    sh(f"git clone -q --depth 1 --branch {BRANCH} --filter=blob:none --sparse {REPO} /content/src && git -C /content/src sparse-checkout set {SUBDIR}")
else:
    sh(f"git clone -q --depth 1 --branch {BRANCH} {REPO} /content/src")
os.chdir(f"/content/src/{SUBDIR}")
print(os.getcwd(), sorted(os.listdir()))

/content/src/colab ['.gitignore', 'README.md', 'build_notebook.py', 'clef_colab.ipynb', 'jevgw', 'models.json', 'pyproject.toml', 'setup_llama.sh', 'tests', 'vendor', 'vendor.sh']


## 2. Which GPU, and which models fit it

In [2]:
import subprocess
from jevgw import catalog

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "NO GPU: Runtime > Change runtime type > GPU")
gpu, mib = catalog.gpu_info()
for m in catalog.load("models.json").values():
    if catalog.runnable(m):
        print(("fits   " if catalog.fits(m, mib) else "NO FIT ") + f"{m['id']:<17} {m['vram_gib']:>5} GiB  {m['name']}")

Tesla T4, 15360 MiB, 580.82.07

fits   clef-flash-q4km    10.8 GiB  Clef Flash Q4_K_M
fits   clef-flash-q2k      9.3 GiB  Clef Flash Q2_K
NO FIT clef-flash         18.5 GiB  Clef Flash (full precision)
fits   laya                3.1 GiB  Laya
fits   verdict               0 GiB  openJev Verdict
fits   semif               8.6 GiB  SemIf
fits   so1                 8.6 GiB  open-alternative-jev
fits   kev-0.5b            2.7 GiB  kev 0.5B
fits   kev-0.8b            4.5 GiB  kev 0.8B
NO FIT kev-4b             17.8 GiB  kev 4B
fits   jeff                9.5 GiB  jeff


## 3. Install the llama.cpp server (Clef Flash Q4 and Q2 use it)
A T4 has no ready-made code in the official build, so its first start compiles it (slow once). For speed, host the prebuilt `llama-b11430-t4.tar.gz` and put its URL below; see the README.

In [3]:
LLAMA_T4_URL = ""  #@param {type:"string"}
import os
os.environ["LLAMA_T4_URL"] = LLAMA_T4_URL
LLAMA = !LLAMA_DIR=$PWD/llama ./setup_llama.sh
LLAMA_BIN = LLAMA[-1]
print(LLAMA_BIN)

/content/src/colab/llama/bin/llama-server


## 4. Start the gateway and open the control panel
The panel loads and unloads models. The first load of a model installs its packages and downloads its weights (minutes).

In [9]:
MODEL = "clef-flash-q4km"  #@param ["clef-flash-q4km", "clef-flash-q2k", "clef-flash", "laya", "verdict", "semif", "so1", "kev-0.5b", "kev-0.8b", "kev-4b", "jeff"]
import json, secrets, subprocess, sys, time, urllib.error, urllib.request, os
from jevgw.client import Client

KEY = secrets.token_urlsafe(12)
URL = "http://127.0.0.1:8000"
gw = subprocess.Popen([sys.executable, "-m", "jevgw", "--port", "8000", "--key", KEY, "--llama-bin", LLAMA_BIN,
                       "--root", os.path.abspath("vendor"), "--work", os.path.abspath("work")],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Wait to see if it starts successfully or crashes
time.sleep(2)
status = gw.poll()
if status is not None:
    print(f"Gateway failed to start immediately with exit code {status}.")
    print("Process Output:")
    print(gw.stdout.read())
else:
    for _ in range(60):
        try:
            urllib.request.urlopen(URL + "/healthz", timeout=2)
        except urllib.error.HTTPError:
            break  # a 503 is fine: it is up, just no model yet
        except OSError:
            time.sleep(1)
    client = Client(URL, KEY)
    from google.colab import output
    output.serve_kernel_port_as_iframe(8000, path=f"/#key={KEY}", height="900")

Gateway failed to start immediately with exit code 1.
Process Output:
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/src/colab/jevgw/__main__.py", line 81, in <module>
    main()
    ~~~~^^
  File "/content/src/colab/jevgw/__main__.py", line 54, in main
    server = serve(app, args.port, args.host)
  File "/content/src/colab/jevgw/server.py", line 226, in serve
    server = ThreadingHTTPServer((host, port), make_handler(app))
  File "/usr/lib/python3.13/socketserver.py", line 457, in __init__
    self.server_bind()
    ~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/http/server.py", line 140, in server_bind
    socketserver.TCPServer.server_bind(self)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/socketserver.py", line 478, in server_bind
    self.socket.bind(self.server_address)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 98] Address alr

In [7]:
import time

# Diagnose why the gateway might not be reachable
if 'gw' in globals():
    status = gw.poll()
    if status is not None:
        print(f"Gateway process has terminated with exit code {status}.")
    else:
        print("Gateway process is still running. Retrying connection...")
else:
    print("Gateway process 'gw' is not defined in the current environment.")

t = time.time()
client.select(MODEL)
print("loaded", MODEL, "in", round(time.time() - t), "s")

Gateway process has terminated with exit code 2.


URLError: <urlopen error [Errno 111] Connection refused>

### Test it
Five timed calls per test after one warm-up. Video goes in as sampled frames. Set `IMAGE` / `VIDEO` to your own files (upload them in the Files panel).

In [ ]:
IMAGE = ""  #@param {type:"string"}
VIDEO = ""  #@param {type:"string"}
from jevgw import client as jc
rows = jc.demo(client, IMAGE or None, VIDEO or None, frames=6, n=5)
for r in rows:
    print(r["test"], "->", json.dumps(r["answers"])[:200])

### Switch to another model
The current one is unloaded first. A model that does not fit this GPU is refused with the reason.

In [ ]:
MODEL = "laya"  #@param ["clef-flash-q4km", "clef-flash-q2k", "clef-flash", "laya", "verdict", "semif", "so1", "kev-0.5b", "kev-0.8b", "kev-4b", "jeff"]
client.select(MODEL)
rows = jc.demo(client, IMAGE or None, VIDEO or None, frames=6, n=5)

## 5. Serve mode (opt-in): a public endpoint
**Read this first.** Colab's FAQ lists, for all runtimes, "web service offerings not related to interactive compute with Colab" and
"connecting to remote proxies", and for free runtimes "bypassing the notebook UI to interact primarily via a web UI". It also says
"Runtimes will time out if you are idle." A public endpoint is the kind of thing those lines describe, and the audio loop is a workaround for the
idle timeout. Google may disconnect the session or restrict your Colab use. It is your account and your call. For anything you want to keep running,
host the gateway on a GPU machine that allows serving (`python -m jevgw --tunnel`).

What this does: opens a free Cloudflare quick tunnel (testing only: no uptime promise, the URL changes if it restarts, 200 requests in flight at most,
no streaming), keeps it alive with a watchdog, and plays a silent audio loop. Every call needs the API key.

In [ ]:
SERVE = False  #@param {type:"boolean"}
if SERVE:
    tunnel = client.tunnel("start")
    sample = {"state": "Our checkout is returning errors.", "questions": {"outage": {"type": "noul", "instructions": "Is a service down?"}}}
    print("endpoint:", tunnel["url"])
    print("key     :", KEY)
    print(f"curl {tunnel['url']}/v1/systemone -H 'Authorization: Bearer {KEY}' -H 'Content-Type: application/json' -d '{json.dumps(sample)}'")
else:
    print("Serve mode is off.")

In [ ]:
# a silent audio loop: some people report it keeps the tab counted as active. Google does not document this, it can stop working,
# and it does not lift the 12 hour limit. Keep this tab open and in the foreground.
import base64, io, wave
from IPython.display import HTML, display
buf = io.BytesIO()
with wave.open(buf, "wb") as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(8000); w.writeframes(b"\x00\x00" * 8000)
display(HTML('<audio autoplay loop controls style="width:100%"><source src="data:audio/wav;base64,' + base64.b64encode(buf.getvalue()).decode() + '"></audio>'))

In [ ]:
# status: run again any time. The URL changes if the tunnel restarted.
print("tunnel:", client.tunnel())
s = client.models()
print("loaded:", s["loaded"], "| load", s["load_s"], "s | warm-up", s["warm_ms"], "ms")
for model, v in client.stats().items():
    print(f"{model:<18} {v['requests']:>5} calls, {v['errors']} errors, p50 {v['p50_ms']} ms, p95 {v['p95_ms']} ms")

## 6. Stop
Frees the GPU and closes the tunnel.

In [ ]:
gw.terminate()